In [ ]:
%matplotlib inline
################################
# 4) Evaluate Edge Classifier
################################
@torch.no_grad()
def evaluate_edge_classifier(model, test_graphs, device='cuda', threshold=0.5):
    model.eval()
    model.to(device)

    all_scores = []
    all_labels = []

    for data in test_graphs:
        data = data.to(device)
        x_dict = model(data)
        z_ip = x_dict['ip']

        for rel_type in data.edge_types:
            if 'edge_label' not in data[rel_type]:
                continue
            labels = data[rel_type].edge_label
            if labels.numel() == 0:
                continue

            edge_index = data[rel_type].edge_index
            src, dst = edge_index
            logits = model.edge_classify(z_ip[src], z_ip[dst])

            probs = F.softmax(logits, dim=-1)[:, 1]
            all_scores.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)

    if len(np.unique(all_labels)) < 2:
        print("Only one class in test set => AUC=0.5 by definition.")
        return

    auc = roc_auc_score(all_labels, all_scores)
    final_preds = (all_scores > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, final_preds, average='binary', zero_division=0
    )
    accuracy = accuracy_score(all_labels, final_preds)
    cm = confusion_matrix(all_labels, final_preds)

    print(f"AUC={auc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, Accuracy={accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Attack"], yticklabels=["Benign", "Attack"])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    plt.show()

    fpr, tpr, _ = roc_curve(all_labels, all_scores)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC={auc:.4f})", lw=2)
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver Operating Characteristic (ROC) Curve")
    plt.legend(loc="lower right")
    plt.savefig("roc_curve.png")
    plt.show()

###############################
# 5) Data-Loading Helpers
###############################
def load_single_graph(directory, filename):
    path = os.path.join(directory, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found.")

    #data = torch.load(path)
    data = torch.load(path, weights_only=False)

    for rel_type in data.edge_types:
        if 'edge_label' in data[rel_type]:
            lbl = data[rel_type].edge_label
            lbl = torch.where(lbl < 0, torch.tensor(0, device=lbl.device), lbl)
            lbl = torch.where(lbl > 1, torch.tensor(1, device=lbl.device), lbl)
            data[rel_type].edge_label = lbl

    return data

#########################
# 6) Main Script
#########################
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    train_dir = "./hetero_graphs_pt"
    train_files = [f for f in os.listdir(train_dir) if f.endswith('.pt')]
    train_graphs = [load_single_graph(train_dir, f) for f in train_files]

    print(f"Loaded {len(train_graphs)} training graphs.")

    hidden_channels = 64
    out_channels = 32
    model = HeteroEdgeClassifier(hidden_channels, out_channels)

    train_edge_classifier(model, train_graphs, epochs=70, device=device)

    test_dir = "./3ed_tes_h_graphs_hetero_graphs"
    test_files = [f for f in os.listdir(test_dir) if f.endswith('.pt')]
    test_graphs = [load_single_graph(test_dir, f) for f in test_files]

    print(f"Loaded {len(test_graphs)} test graph(s).")

    best_threshold = 0.01
    evaluate_edge_classifier(model, test_graphs, device=device, threshold=best_threshold)


In [ ]:
%matplotlib inline

################################
# 4) Evaluate Edge Classifier (CON DIAGNOSTICA INTEGRATA)
################################
@torch.no_grad()
def evaluate_edge_classifier(model, test_graphs, device='cuda', threshold=0.5):
    model.eval()
    model.to(device)

    all_scores = []
    all_labels = []

    for data in test_graphs:
        data = data.to(device)
        x_dict = model(data)
        z_ip = x_dict['ip']

        for rel_type in data.edge_types:
            if 'edge_label' not in data[rel_type]:
                continue
            labels = data[rel_type].edge_label
            if labels.numel() == 0:
                continue

            edge_index = data[rel_type].edge_index
            src, dst = edge_index
            logits = model.edge_classify(z_ip[src], z_ip[dst])

            probs = F.softmax(logits, dim=-1)[:, 1]
            all_scores.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)

    if len(np.unique(all_labels)) < 2:
        print("Only one class in test set => AUC=0.5 by definition.")
        return

    # --- INIZIO BLOCCO DIAGNOSTICA ---
    attack_scores = all_scores[all_labels == 1]
    benign_scores = all_scores[all_labels == 0]
    
    print("\n" + "="*40)
    print("🔬 COSA STA SUCCEDENDO DIETRO LE QUINTE:")
    print(f"-> Numero totale di flussi Benigni nel Test: {len(benign_scores)}")
    print(f"-> Numero totale di flussi di Attacco nel Test: {len(attack_scores)}")
    print(f"-> I 6 punteggi reali degli attacchi: {np.sort(attack_scores)}")
    print(f"-> Punteggio massimo assegnato a un flusso Benign: {np.max(benign_scores):.8f}")
    
    # Calcolo della soglia geometrica ottimale (Indice di Youden)
    fpr, tpr, thresholds = roc_curve(all_labels, all_scores)
    idx = np.argmax(tpr - fpr)
    soglia_ottimale = thresholds[idx]
    print(f"-> 🎯 La vera soglia matematica scoperta è: {soglia_ottimale}")
    print("="*40 + "\n")
    
    # Sovrascriviamo la soglia passata con quella dinamica ottimale
    threshold = soglia_ottimale
    # --- FINE BLOCCO DIAGNOSTICA ---

    auc = roc_auc_score(all_labels, all_scores)
    final_preds = (all_scores > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, final_preds, average='binary', zero_division=0
    )
    accuracy = accuracy_score(all_labels, final_preds)
    cm = confusion_matrix(all_labels, final_preds)

    print(f"RISULTATI FINALI CON SOGLIA OTTIMIZZATA:")
    print(f"AUC={auc:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}, Accuracy={accuracy:.4f}")
    print("Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Benign", "Attack"], yticklabels=["Benign", "Attack"])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix")
    plt.savefig("confusion_matrix.png")
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"ROC Curve (AUC={auc:.4f})", lw=2)
    plt.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Receiver Operating Characteristic (ROC) Curve")
    plt.legend(loc="lower right")
    plt.savefig("roc_curve.png")
    plt.show()

###############################
# 5) Data-Loading Helpers
###############################
def load_single_graph(directory, filename):
    path = os.path.join(directory, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found.")

    data = torch.load(path, weights_only=False)

    for rel_type in data.edge_types:
        if 'edge_label' in data[rel_type]:
            lbl = data[rel_type].edge_label
            lbl = torch.where(lbl < 0, torch.tensor(0, device=lbl.device), lbl)
            lbl = torch.where(lbl > 1, torch.tensor(1, device=lbl.device), lbl)
            data[rel_type].edge_label = lbl

    return data

#########################
# 6) Main Script
#########################
if __name__ == "__main__":
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")

    train_dir = "./hetero_graphs_pt"
    train_files = [f for f in os.listdir(train_dir) if f.endswith('.pt')]
    train_graphs = [load_single_graph(train_dir, f) for f in train_files]

    print(f"Loaded {len(train_graphs)} training graphs.")

    hidden_channels = 64
    out_channels = 32
    model = HeteroEdgeClassifier(hidden_channels, out_channels)

    train_edge_classifier(model, train_graphs, epochs=70, device=device)

    test_dir = "./3ed_tes_h_graphs_hetero_graphs"
    test_files = [f for f in os.listdir(test_dir) if f.endswith('.pt')]
    test_graphs = [load_single_graph(test_dir, f) for f in test_files]

    print(f"Loaded {len(test_graphs)} test graph(s).")

    # Questo valore ora verrà corretto dinamicamente nel testo sopra
    evaluate_edge_classifier(model, test_graphs, device=device, threshold=0.01)